In [ ]:
## Colab Setup (skip if running locally)
# Run the cell only on Google Colab.

!git config --global user.email 'andrexnardo80@gmail.com'
!git config --global user.name 'flaviofrasca'

import os

!git clone https://github.com/flaviofrasca/MaskArchitectureAnomaly_CourseProject.git
os.chdir('/content/MaskArchitectureAnomaly_CourseProject/eomt')
print(os.getcwd())

from google.colab import drive
drive.mount('/content/drive')

!pip install -q \
    "lightning==2.5.1.post0" \
    "timm==1.0.15" \
    "transformers==4.56.1" \
    "torchmetrics==1.7.1" \
    "jsonargparse[signatures]==4.38" \
    "pycocotools==2.0.8" \
    "fvcore==0.1.5.post20221221" \
    "wandb==0.19.10" \
    "scipy==1.15.2" \
    "gitignore_parser==0.1.12"

In [ ]:
import os, glob, torch
os.environ["WANDB_MODE"] = "disabled"

BASE = "/content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared"
COCO_CKPT = BASE + "/checkpoints/coco/eomt_coco.bin"
DATA_PATH = BASE + "/datasets/cityscapes"
SAVE_DIR  = BASE + "/checkpoints/finetuned"
CONFIG    = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"

def extract_weights(ckpt_path, out_path):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state_dict = ckpt.get("state_dict", ckpt)
    torch.save(state_dict, out_path)
    print(f"Weights saved → {out_path}")
    return out_path

def find_last_checkpoint(phase_dir):
    ckpts = glob.glob(f"{phase_dir}/**/*.ckpt", recursive=True)
    if not ckpts:
        raise FileNotFoundError(f"Nessun checkpoint in {phase_dir}")
    return sorted(ckpts, key=os.path.getmtime)[-1]

In [ ]:
phase_dir = f"{SAVE_DIR}/phase1_head_only"
os.makedirs(phase_dir, exist_ok=True)

!python main.py fit \
  --config {CONFIG} \
  --model.init_args.ckpt_path {COCO_CKPT} \
  --model.init_args.load_ckpt_class_head False \
  --model.init_args.llrd 0.0 \
  --model.init_args.lr_mult 0.0 \
  --model.init_args.attn_mask_annealing_enabled False \
  --data.init_args.path {DATA_PATH} \
  --data.init_args.batch_size 4 \
  --data.init_args.num_workers 2 \
  --trainer.max_epochs 10 \
  --trainer.default_root_dir {phase_dir} \
  "--trainer.callbacks+={\"class_path\": \"lightning.pytorch.callbacks.ModelCheckpoint\", \"init_args\": {\"save_last\": true, \"every_n_epochs\": 1}}" \
  --compile_disabled

PHASE1_CKPT = extract_weights(find_last_checkpoint(phase_dir), f"{phase_dir}/weights.bin")
print(f"Phase 1 done → {PHASE1_CKPT}")

In [ ]:
phase_dir = f"{SAVE_DIR}/phase2_unfreeze_last"
os.makedirs(phase_dir, exist_ok=True)

!python main.py fit \
  --config {CONFIG} \
  --model.init_args.ckpt_path {PHASE1_CKPT} \
  --model.init_args.load_ckpt_class_head True \
  --model.init_args.llrd 0.3 \
  --model.init_args.attn_mask_annealing_enabled False \
  --data.init_args.path {DATA_PATH} \
  --data.init_args.batch_size 4 \
  --data.init_args.num_workers 2 \
  --trainer.max_epochs 5 \
  --trainer.default_root_dir {phase_dir} \
  "--trainer.callbacks+={\"class_path\": \"lightning.pytorch.callbacks.ModelCheckpoint\", \"init_args\": {\"save_last\": true, \"every_n_epochs\": 1}}" \
  --compile_disabled

PHASE2_CKPT = extract_weights(find_last_checkpoint(phase_dir), f"{phase_dir}/weights.bin")
print(f"Phase 2 done → {PHASE2_CKPT}")

In [ ]:
phase_dir = f"{SAVE_DIR}/phase3_unfreeze_more"
os.makedirs(phase_dir, exist_ok=True)

!python main.py fit \
  --config {CONFIG} \
  --model.init_args.ckpt_path {PHASE2_CKPT} \
  --model.init_args.load_ckpt_class_head True \
  --model.init_args.llrd 0.6 \
  --model.init_args.attn_mask_annealing_enabled False \
  --data.init_args.path {DATA_PATH} \
  --data.init_args.batch_size 4 \
  --data.init_args.num_workers 2 \
  --trainer.max_epochs 5 \
  --trainer.default_root_dir {phase_dir} \
  "--trainer.callbacks+={\"class_path\": \"lightning.pytorch.callbacks.ModelCheckpoint\", \"init_args\": {\"save_last\": true, \"every_n_epochs\": 1}}" \
  --compile_disabled

PHASE3_CKPT = extract_weights(find_last_checkpoint(phase_dir), f"{phase_dir}/weights.bin")
print(f"Phase 3 done → {PHASE3_CKPT}")

In [ ]:
phase_dir = f"{SAVE_DIR}/phase4_full"
os.makedirs(phase_dir, exist_ok=True)

!python main.py fit \
  --config {CONFIG} \
  --model.init_args.ckpt_path {PHASE3_CKPT} \
  --model.init_args.load_ckpt_class_head True \
  --model.init_args.llrd 0.8 \
  --model.init_args.attn_mask_annealing_enabled False \
  --data.init_args.path {DATA_PATH} \
  --data.init_args.batch_size 4 \
  --data.init_args.num_workers 2 \
  --trainer.max_epochs 5 \
  --trainer.default_root_dir {phase_dir} \
  "--trainer.callbacks+={\"class_path\": \"lightning.pytorch.callbacks.ModelCheckpoint\", \"init_args\": {\"save_last\": true, \"every_n_epochs\": 1}}" \
  --compile_disabled

PHASE4_CKPT = extract_weights(find_last_checkpoint(phase_dir), f"{phase_dir}/weights.bin")
print(f"Phase 4 done → {PHASE4_CKPT}")